<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/project-2-llm-evaluation-suite/04b_judge_alignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 4b: Judge Alignment: Calibration and Post-Calibration Score

**Goal:** Address the Phase 4a disagreement (as_006: human BORDERLINE,
judge PASS) by adding a coverage_completeness aspect to the AspectCritic
rubric. Re-run the full alignment evaluation with the six-aspect set and
document the post-calibration alignment score. The alignment score is the
metric that validates the Claude judge before it is trusted in production
governance evaluation.

**Tools:** RAGAS AspectCritic (extended), Claude (claude-sonnet-4-6) as judge

**The Phase 4a finding this notebook addresses:**
AspectCritic evaluates compliance accuracy. It does not evaluate coverage
completeness. A partial but accurate response passes all five accuracy
aspects correctly but a human reviewer flags it as borderline because a
deployer acting on the response alone would lack sufficient information.
The fix is a sixth aspect: coverage_completeness.

**Design addition (Federico Blanco Sanchez-Llanos):** The G-Eval compliance
verdict is exported as a signed artifact bound to a hash of the specific
inputs. Acknowledged here: this requirement emerged from the LinkedIn
exchange and is built independently using Python hashlib rather than
any external vendor endpoint.

**SIMULATED_OUTPUT flag:** Set to True throughout.

**Date:** July 2026

In [1]:
# Cell 2: Mount Drive and confirm Phase 4a

from google.colab import drive
drive.mount('/content/drive')

import os, json

DRIVE_PATH = "/content/drive/MyDrive/python-ai-governance-p2/data/"

phase4a_path = DRIVE_PATH + "phase04a_aspect_critic_results.json"
if os.path.exists(phase4a_path):
    with open(phase4a_path) as f:
        phase4a = json.load(f)
    print("Phase 4a results confirmed.")
    print(f"  Pre-calibration alignment: "
          f"{phase4a['alignment']['pre_calibration_score']:.1%}")
    print(f"  Agreements: {phase4a['alignment']['agreements']}/"
          f"{phase4a['alignment']['total']}")
    print(f"  Disagreements: "
          f"{len(phase4a['alignment']['disagreements'])}")
    for d in phase4a["alignment"]["disagreements"]:
        print(f"    {d['id']}: human {d['human_label']} vs "
              f"judge {d['judge_verdict']}")
        print(f"    Root cause: {d['root_cause'][:80]}...")
else:
    print("WARNING: Phase 4a results not found.")
    print(f"Expected: {phase4a_path}")
    print("Run 04a_ragas_aspect_critic.ipynb first.")

Mounted at /content/drive
Phase 4a results confirmed.
  Pre-calibration alignment: 83.3%
  Agreements: 5/6
  Disagreements: 1
    as_006: human BORDERLINE vs judge PASS
    Root cause: Coverage completeness vs compliance accuracy. AspectCritic evaluates whether cla...


In [2]:
# Cell 3: Install packages

!pip install ragas==0.3.9 langfuse anthropic \
    google-generativeai langchain-google-genai \
    langchain-community langchain-google-vertexai --quiet

print("Packages installed.")
print("ragas==0.3.9 (pinned: avoids broken VertexAI import in 0.4.x)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.7/366.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5

In [3]:
# Cell 4: Simulated output flag, clients, and thresholds

SIMULATED_OUTPUT = True

JUDGE_MODEL = "claude-sonnet-4-6"

from google.colab import userdata

if not SIMULATED_OUTPUT:
    import anthropic
    claude_client = anthropic.Anthropic(
        api_key=userdata.get('ANTHROPIC_API_KEY')
    )
    from langfuse import Langfuse
    langfuse = Langfuse(
        public_key=userdata.get('LANGFUSE_PUBLIC_KEY'),
        secret_key=userdata.get('LANGFUSE_SECRET_KEY'),
        host="https://cloud.langfuse.com"
    )
    print("Claude client initialised.")
    print("Langfuse client initialised.")
else:
    print("[SIMULATED] Clients not initialised.")
    print(f"SIMULATED_OUTPUT = {SIMULATED_OUTPUT}")
    print(f"Judge model: {JUDGE_MODEL}")

PASS_THRESHOLD = 0.80
FAIL_THRESHOLD = 0.60

VERDICT_SCORES = {"yes": 1.0, "no": 0.0}

print()
print("Routing thresholds consistent across all phases:")
print(f"  >= {PASS_THRESHOLD}: PASS      -> quality layer")
print(f"  <  {FAIL_THRESHOLD}: FAIL      -> governance layer")
print(f"  between:    BORDERLINE -> human review queue")

[SIMULATED] Clients not initialised.
SIMULATED_OUTPUT = True
Judge model: claude-sonnet-4-6

Routing thresholds consistent across all phases:
  >= 0.8: PASS      -> quality layer
  <  0.6: FAIL      -> governance layer
  between:    BORDERLINE -> human review queue


In [4]:
# Cell 5: Restore knowledge base and pipeline

REGULATORY_DOCS = {
    "doc_001": {
        "title": "EU AI Act Article 10: Data Governance",
        "content": (
            "Article 10 requires that high-risk AI systems use training, validation "
            "and testing data subject to data governance practices. Data sets must be "
            "relevant, representative, and free of errors. Providers must examine data "
            "for possible biases. Special category data may only be used under specific "
            "conditions to detect and correct bias. Disparate impact ratios below 0.80 "
            "indicate a potential Article 10 violation."
        )
    },
    "doc_002": {
        "title": "EU AI Act Article 14: Human Oversight",
        "content": (
            "Article 14 requires high-risk AI systems to be designed to allow effective "
            "human oversight during use. Persons assigned to oversight must understand "
            "the system's capacities and limitations, monitor its operation, intervene "
            "or interrupt it when necessary, and not be unduly influenced to over-rely "
            "on its outputs. Non-compliance: up to EUR 15 million or 3 percent of "
            "global annual turnover under Article 99(3)."
        )
    },
    "doc_003": {
        "title": "NIST AI RMF: GOVERN Function",
        "content": (
            "The GOVERN function establishes the policies, processes, and procedures "
            "required for AI risk management across the organisation. It includes "
            "assigning accountability for AI risks, establishing a culture of risk "
            "awareness, and ensuring that AI governance is integrated into existing "
            "enterprise risk management frameworks."
        )
    },
    "doc_004": {
        "title": "EU AI Act Article 99: Penalties",
        "content": (
            "Article 99 establishes a three-tier penalty structure. "
            "Tier 1: violations of prohibited AI practices under Article 5 "
            "carry penalties up to EUR 35 million or 7 percent of global turnover. "
            "Tier 2: violations of high-risk AI obligations carry penalties "
            "up to EUR 15 million or 3 percent of global turnover. "
            "Tier 3: incorrect information to authorities carries penalties "
            "up to EUR 7.5 million or 1 percent of global turnover."
        )
    },
    "doc_005": {
        "title": "ISO/IEC 42001: AI Management System",
        "content": (
            "ISO/IEC 42001 specifies requirements for establishing, implementing, "
            "maintaining and continually improving an AI management system. "
            "Clause 8 requires organisations to plan, implement, control, and review "
            "processes needed to meet AI system impact requirements. "
            "Clause 9 requires performance evaluation through monitoring, "
            "measurement, analysis and evaluation."
        )
    }
}


def retrieve_documents(query: str, n_results: int = 2) -> list:
    if SIMULATED_OUTPUT:
        q = query.lower()
        if "oversight" in q or "human" in q or "article 14" in q:
            return [
                {"id": "doc_002",
                 "title": REGULATORY_DOCS["doc_002"]["title"],
                 "content": REGULATORY_DOCS["doc_002"]["content"],
                 "distance": 0.12},
                {"id": "doc_004",
                 "title": REGULATORY_DOCS["doc_004"]["title"],
                 "content": REGULATORY_DOCS["doc_004"]["content"],
                 "distance": 0.24},
            ]
        elif "data" in q or "bias" in q or "article 10" in q:
            return [
                {"id": "doc_001",
                 "title": REGULATORY_DOCS["doc_001"]["title"],
                 "content": REGULATORY_DOCS["doc_001"]["content"],
                 "distance": 0.11},
                {"id": "doc_002",
                 "title": REGULATORY_DOCS["doc_002"]["title"],
                 "content": REGULATORY_DOCS["doc_002"]["content"],
                 "distance": 0.31},
            ]
        elif "nist" in q or "govern" in q or "rmf" in q:
            return [
                {"id": "doc_003",
                 "title": REGULATORY_DOCS["doc_003"]["title"],
                 "content": REGULATORY_DOCS["doc_003"]["content"],
                 "distance": 0.09},
                {"id": "doc_001",
                 "title": REGULATORY_DOCS["doc_001"]["title"],
                 "content": REGULATORY_DOCS["doc_001"]["content"],
                 "distance": 0.38},
            ]
        elif "penalty" in q or "article 99" in q or "fine" in q:
            return [
                {"id": "doc_004",
                 "title": REGULATORY_DOCS["doc_004"]["title"],
                 "content": REGULATORY_DOCS["doc_004"]["content"],
                 "distance": 0.08},
                {"id": "doc_002",
                 "title": REGULATORY_DOCS["doc_002"]["title"],
                 "content": REGULATORY_DOCS["doc_002"]["content"],
                 "distance": 0.33},
            ]
        else:
            return [
                {"id": "doc_002",
                 "title": REGULATORY_DOCS["doc_002"]["title"],
                 "content": REGULATORY_DOCS["doc_002"]["content"],
                 "distance": 0.18},
                {"id": "doc_003",
                 "title": REGULATORY_DOCS["doc_003"]["title"],
                 "content": REGULATORY_DOCS["doc_003"]["content"],
                 "distance": 0.29},
            ]
    results = collection.query(query_texts=[query], n_results=n_results)
    return [
        {
            "id": results["ids"][0][i],
            "title": results["metadatas"][0][i]["title"],
            "content": results["documents"][0][i],
            "distance": results["distances"][0][i]
        }
        for i in range(len(results["ids"][0]))
    ]


def generate_response(query: str, retrieved_docs: list) -> dict:
    context = "\n\n".join(
        f"[{d['title']}]\n{d['content']}" for d in retrieved_docs
    )
    if SIMULATED_OUTPUT:
        q = query.lower()
        if "oversight" in q or "human" in q or "article 14" in q:
            response_text = (
                "Based on EU AI Act Article 14, high-risk AI systems must be "
                "designed to allow effective human oversight. Persons assigned "
                "to oversight must understand the system's capacities and "
                "limitations, monitor its operation, and intervene or interrupt "
                "it when necessary. Non-compliance carries penalties of up to "
                "EUR 15 million or 3 percent of global annual turnover."
            )
        elif "data" in q or "bias" in q or "article 10" in q:
            response_text = (
                "Under EU AI Act Article 10, high-risk AI systems must use "
                "training, validation and testing data subject to data governance "
                "practices. Data sets must be relevant, representative, and free "
                "of errors. Providers must examine data for possible biases. "
                "Disparate impact ratios below 0.80 indicate a potential "
                "Article 10 violation."
            )
        elif "penalty" in q or "article 99" in q:
            response_text = (
                "Article 99 establishes a three-tier penalty structure. "
                "Tier 1 carries penalties up to EUR 35 million or 7 percent "
                "of global annual turnover. Tier 2 carries penalties up to "
                "EUR 15 million or 3 percent for high-risk AI violations."
            )
        elif "nist" in q or "govern" in q:
            response_text = (
                "The NIST AI RMF GOVERN function requires organisations to "
                "establish policies, processes, and procedures for AI risk "
                "management, including assigning accountability and integrating "
                "governance into enterprise risk management frameworks."
            )
        else:
            response_text = (
                "Based on the retrieved regulatory documents, the system must "
                "comply with applicable AI governance requirements."
            )
        return {
            "query": query,
            "retrieved_doc_ids": [d["id"] for d in retrieved_docs],
            "response": response_text,
            "model": "gemini-flash-latest",
            "simulated": True
        }
    response = gemini_client.models.generate_content(
        model="gemini-flash-latest",
        contents=prompt
    )
    return {
        "query": query,
        "retrieved_doc_ids": [d["id"] for d in retrieved_docs],
        "response": response.text,
        "model": "gemini-flash-latest",
        "simulated": False
    }


print("REGULATORY_DOCS restored.")
print("retrieve_documents() restored.")
print("generate_response() restored.")

REGULATORY_DOCS restored.
retrieve_documents() restored.
generate_response() restored.


In [5]:
# Cell 6: Extended aspect set with coverage_completeness

import hashlib
import json
from datetime import datetime

# The five Phase 4a aspects are carried forward unchanged.
# A sixth aspect is added to address the as_006 disagreement:
# coverage_completeness captures whether the response provides
# sufficient information for a deployer to act on it, not just
# whether the claims it makes are accurate.
#
# This is the calibration step: adding the missing aspect,
# re-running alignment, and documenting the improvement.

ASPECTS_V1 = [
    {
        "name": "correctness",
        "definition": (
            "The response contains only factually accurate claims that "
            "are directly supported by the retrieved regulatory documents. "
            "No figures, thresholds, or obligations are misrepresented."
        )
    },
    {
        "name": "regulatory_grounding",
        "definition": (
            "Every regulatory claim in the response can be traced to a "
            "specific article, section, or provision in the retrieved "
            "documents. The response does not introduce regulatory "
            "obligations that are not present in the retrieved context."
        )
    },
    {
        "name": "oversight_representation",
        "definition": (
            "If human oversight is discussed, it is represented as a "
            "mandatory requirement under Article 14, not a recommendation. "
            "The response does not suggest that automated monitoring alone "
            "satisfies the Article 14 human oversight obligation."
        )
    },
    {
        "name": "bias_representation",
        "definition": (
            "If data governance or bias is discussed, the 0.80 disparate "
            "impact ratio threshold is correctly represented. The response "
            "does not suggest that bias examination is optional for "
            "high-risk AI systems under Article 10."
        )
    },
    {
        "name": "harm_potential",
        "definition": (
            "The response does not risk misleading a deployer about their "
            "compliance obligations in a way that could result in "
            "non-compliance with the EU AI Act or NIST AI RMF. "
            "Verdict: yes means low harm potential (good). "
            "No means the response could mislead a deployer (bad)."
        )
    }
]

# Sixth aspect: the calibration addition
COVERAGE_COMPLETENESS_ASPECT = {
    "name": "coverage_completeness",
    "definition": (
        "The response provides sufficient information for a deployer "
        "to understand and act on their compliance obligation. "
        "A response that is accurate but omits critical implementation "
        "details, specific thresholds, or actionable steps that are "
        "present in the retrieved documents does not satisfy this aspect. "
        "Verdict: yes means the response is actionably complete. "
        "No means a deployer would need to seek additional information "
        "before they could act on the response."
    )
}

ASPECTS_V2 = ASPECTS_V1 + [COVERAGE_COMPLETENESS_ASPECT]

print(f"Phase 4a aspects (v1): {len(ASPECTS_V1)}")
print(f"Phase 4b aspects (v2): {len(ASPECTS_V2)}")
print()
print("New aspect added:")
print(f"  {COVERAGE_COMPLETENESS_ASPECT['name']}:")
print(f"  {COVERAGE_COMPLETENESS_ASPECT['definition'][:120]}...")
print()
print("Rationale: as_006 (NIST GOVERN partial response) was human-labeled")
print("BORDERLINE because the response is accurate but incomplete.")
print("None of the five v1 aspects captured coverage completeness.")
print("The v2 aspect set addresses this gap directly.")

Phase 4a aspects (v1): 5
Phase 4b aspects (v2): 6

New aspect added:
  coverage_completeness:
  The response provides sufficient information for a deployer to understand and act on their compliance obligation. A resp...

Rationale: as_006 (NIST GOVERN partial response) was human-labeled
BORDERLINE because the response is accurate but incomplete.
None of the five v1 aspects captured coverage completeness.
The v2 aspect set addresses this gap directly.
